In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score, accuracy_score

import json

In [2]:
data = pd.read_csv('../data/FINAL_DATA.csv')
X = data['Clean_Reviews'].astype(str)
y = data['Sentiment_label'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size = 0.2,
                                                    random_state = 42,
                                                    stratify = y)

print(f'Training Data length: {X_train} | Testing data length: {X_test}')



Training Data length: 1150                                           bad quality
19108    munasib qeemat mein bohat achi hai packing bhi...
14448    this time everything was really bad quality a ...
74           got the delivery on time and in one go thanks
19812                            yeh aik acha masnoaat hai
                               ...                        
5595     very great experience with daraz this is my fi...
15638    nice bags also seller is good recommended this...
12174         it not pairing with any device wast of money
4655                         i love these such a great fit
18630                                    behtareen product
Name: Clean_Reviews, Length: 16730, dtype: str | Testing data length: 5362     product packing was done nicely came in a nice...
2748     very fast delivery in too much economical pric...
16420           red order kia woi mila higly recommendable
592      doesn t work connects and disconnect not readi...
4837     the quality is

In [3]:
"""
BASELINE MODELS TRAINING
1. VADER BASELINE
2. TF-IDF + LOGISTIC REGRESSION
3. TF-IDF + SVM

>> VADER BASELINE MODEL:
   - A rule based sentiment analysis model where no training is required
   - Work using a predefined sentiment lexicon 
   (have scores for each word in dictionary, then add them up according to the sentence)

"""
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer() # loads predefined sentiment lexicon

def vader_apply(text):
    score = analyzer.polarity_scores(str(text))['compound'] # compund score range : -1 to +1 (overall sentiment of the whole sentence)
    # class mapping
    if score >= 0.05: # positive
        return 2
    elif score <= -0.05: # negative
        return 0
    else:  # neutral (-0.05, 0.05) -> default vader thresholds
        return 1
    

vader_preds = X_test.apply(vader_apply)
vader_acc = accuracy_score(y_test, vader_preds)
vader_f1 = f1_score(y_test, vader_preds, average='macro') # computes f1 score for each class separately then take avg (due to class imbalanced)

print("Vader Results: ")
print(f"1. ACCURACY SCORE: {vader_acc}\n2. F1-SCORE: {vader_f1}")

# printing the classfication report for the vader baseline model analysis
print(classification_report(y_test, vader_preds,
                            target_names=['Negative', 'Neutral', 'Positve']))
# support = number of samples per class


Vader Results: 
1. ACCURACY SCORE: 0.633277551996175
2. F1-SCORE: 0.5331076862574706
              precision    recall  f1-score   support

    Negative       0.78      0.52      0.63      1069
     Neutral       0.16      0.26      0.20       606
     Positve       0.78      0.77      0.78      2508

    accuracy                           0.63      4183
   macro avg       0.57      0.52      0.53      4183
weighted avg       0.69      0.63      0.65      4183



In [4]:
"""
>> TF-IDF + LOGISTIC REGRESSION
VADER had limitations; rule based and was poor on this current data especially for positive class
So now switching to TF-IDF and Logistic Regression

TF (Term Frequency) -> how often word appears
IDF (Inverse Document Frequency) -> how important the word is

- converts text -> numbers (give importance to those words that are rare in the document)
- model learn patterns from our data (training)

"""
# Handle missing values
X_train = X_train.fillna("")
X_test = X_test.fillna("")

tfidf = TfidfVectorizer(max_features=15000, ngram_range=(1,2), sublinear_tf=True)
# 15000 top important words along with single words + pairs of words and using log scaling

X_train_tfidf = tfidf.fit_transform(X_train) # learn tarining data
X_test_tfidf = tfidf.transform(X_test) # use same vocab, doesnt learn anything new

log_reg_model = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
log_reg_model.fit(X_train_tfidf, y_train) #now model learns which words are +ve, -ve and neutral

log_reg_preds = log_reg_model.predict(X_test_tfidf)

lr_acc = accuracy_score(y_test, log_reg_preds)
lr_f1score = f1_score(y_test, log_reg_preds, average='macro')


print("TF-idf + Logistic Regression Results: ")
print(f"1. ACCURACY SCORE: {lr_acc}\n2. F1-SCORE: {lr_f1score}")

# printing the classfication report for the tfidf + logistic regression model analysis
print(classification_report(y_test, log_reg_preds,
                            target_names=['Negative', 'Neutral', 'Positve']))
# support = number of samples per class



TF-idf + Logistic Regression Results: 
1. ACCURACY SCORE: 0.8646904135787712
2. F1-SCORE: 0.7974697361698563
              precision    recall  f1-score   support

    Negative       0.84      0.83      0.83      1069
     Neutral       0.78      0.54      0.64       606
     Positve       0.89      0.96      0.92      2508

    accuracy                           0.86      4183
   macro avg       0.84      0.78      0.80      4183
weighted avg       0.86      0.86      0.86      4183



In [5]:
"""
>> TF-IDF + SVM 
- SVM finds the best decision boundary/Hyperplane that separates:
  Negative | Positive | Neutral

"""

svm = LinearSVC(max_iter=2000, C=0.5, random_state = 42)
svm.fit(X_train_tfidf, y_train) # find boundaries between sentiment classes
svm_preds = svm.predict(X_test_tfidf)
svm_f1 = f1_score(y_test,  svm_preds, average = 'macro')
svm_acc = accuracy_score(y_test, svm_preds)

print("TF-idf + SVM Results: ")
print(f"1. ACCURACY SCORE: {svm_acc}\n2. F1-SCORE: {svm_f1}")

# printing the classfication report for the tfidf + SVM model analysis
print(classification_report(y_test, svm_preds,
                            target_names=['Negative', 'Neutral', 'Positive']))
# support = number of samples per class


TF-idf + SVM Results: 
1. ACCURACY SCORE: 0.8608654076021994
2. F1-SCORE: 0.7944965017669475
              precision    recall  f1-score   support

    Negative       0.83      0.83      0.83      1069
     Neutral       0.74      0.55      0.63       606
    Positive       0.90      0.95      0.92      2508

    accuracy                           0.86      4183
   macro avg       0.82      0.78      0.79      4183
weighted avg       0.86      0.86      0.86      4183



In [6]:
baseline_results = {
    'VADER': {
        'accuracy': round(vader_acc, 4),
        'f1_macro': round(vader_f1, 4)
    },
    'TF-IDF+LR': {
        'accuracy': round(lr_acc, 4),
        'f1_macro': round(lr_f1score, 4)
    },
    'TF-IDF+SVM': {
        'accuracy': round(svm_acc, 4),
        'f1_macro': round(svm_f1, 4)
    }
}

with open('../results/baseline_results.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)

In [7]:
pd.DataFrame({
    'text': X_test,
    'true_label': y_test
}).to_csv('../results/test_set.csv', index=False)

print("Baseline results saved!")

Baseline results saved!
